Celda 1 — Imports y rutas

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate

from sklearn.ensemble import RandomForestClassifier
import joblib

BASE_DIR = r"C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 2"
FEATURES_CSV = r"C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 1\features_dataset_v1.csv"

OUT_MODEL  = os.path.join(BASE_DIR, "RANDOMFOREST_model.joblib")
OUT_REPORT = os.path.join(BASE_DIR, "RANDOMFOREST_rf_report.txt")

print("BASE_DIR:", BASE_DIR)
print("FEATURES_CSV:", FEATURES_CSV)

BASE_DIR: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 2
FEATURES_CSV: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 1\features_dataset_v1.csv


Celda 2 — Cargar dataset + limpieza mínima (igual que LogReg)

In [2]:
df = pd.read_csv(FEATURES_CSV)

# Capear outliers
if "loudness_var_proxy_db" in df.columns:
    df["loudness_var_proxy_db"] = df["loudness_var_proxy_db"].clip(upper=40)

# Target
y = df["y_bin"].astype(int)

# Selección de features
non_feature_cols = ["clip_id", "file_name", "start_time_s", "end_time_s", "qc_result", "y_bin"]
feature_cols = [c for c in df.columns if c not in non_feature_cols]

# Limpieza de NaN / inf
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

X = df[feature_cols].copy()

print("Filas:", len(df))
print("Features:", feature_cols)
print("Distribución y_bin:\n", y.value_counts())

Filas: 764
Features: ['rms_mean', 'crest_factor_db', 'true_peak_dbfs', 'short_term_lufs_mean', 'loudness_var_proxy_db', 'silence_ratio', 'stereo_correlation', 'spectral_centroid_mean', 'spectral_flatness_mean', 'clipping_ratio', 'near_ceiling_ratio']
Distribución y_bin:
 y_bin
0    399
1    365
Name: count, dtype: int64


Celda 3 — Split Train/Val/Test (mismo split que LogReg)

In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))

Train: 534 Val: 115 Test: 115


Celda 4 — Entrenar Random Forest (modelo comparativo)

In [4]:
rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
    max_depth=8,
    min_samples_leaf=4,
    min_samples_split=8,
    bootstrap=True,
    oob_score=True
)

rf.fit(X_train, y_train)
print("Entrenamiento RF listo.")
print("OOB score:", rf.oob_score_)

Entrenamiento RF listo.
OOB score: 0.9737827715355806


In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_validate(
    rf,
    X,
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "f1": "f1",
        "roc_auc": "roc_auc",
        "recall": "recall"
    },
    n_jobs=-1,
    return_train_score=False
)

print("=== VALIDACIÓN CRUZADA 5-FOLD ===")
print("Accuracy mean:", np.mean(cv_scores["test_accuracy"]))
print("Accuracy std :", np.std(cv_scores["test_accuracy"]))
print("F1 mean      :", np.mean(cv_scores["test_f1"]))
print("F1 std       :", np.std(cv_scores["test_f1"]))
print("Recall mean  :", np.mean(cv_scores["test_recall"]))
print("Recall std   :", np.std(cv_scores["test_recall"]))
print("ROC AUC mean :", np.mean(cv_scores["test_roc_auc"]))
print("ROC AUC std  :", np.std(cv_scores["test_roc_auc"]))

=== VALIDACIÓN CRUZADA 5-FOLD ===
Accuracy mean: 0.9816907464740282
Accuracy std : 0.015679832668425535
F1 mean      : 0.980763144901076
F1 std       : 0.016328100341040437
Recall mean  : 0.9753424657534246
Recall std   : 0.015975210670809054
ROC AUC mean : 0.9981844980058956
ROC AUC std  : 0.0022364837270279912


Celda 5 — Evaluación (Val + Test)

In [6]:
def eval_split(name, model, X_split, y_split):
    pred = model.predict(X_split)
    proba = model.predict_proba(X_split)[:, 1]

    cm = confusion_matrix(y_split, pred)
    rep = classification_report(y_split, pred, digits=4)
    auc = roc_auc_score(y_split, proba)

    print(f"\n=== {name} ===")
    print("Confusion Matrix:\n", cm)
    print("\nClassification Report:\n", rep)
    print("ROC AUC:", round(auc, 4))

    return cm, rep, auc

cm_val, rep_val, auc_val = eval_split("VALIDACIÓN", rf, X_val, y_val)
cm_test, rep_test, auc_test = eval_split("TEST", rf, X_test, y_test)


=== VALIDACIÓN ===
Confusion Matrix:
 [[60  0]
 [ 1 54]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9836    1.0000    0.9917        60
           1     1.0000    0.9818    0.9908        55

    accuracy                         0.9913       115
   macro avg     0.9918    0.9909    0.9913       115
weighted avg     0.9914    0.9913    0.9913       115

ROC AUC: 0.9997

=== TEST ===
Confusion Matrix:
 [[60  0]
 [ 0 55]]

Classification Report:
               precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        60
           1     1.0000    1.0000    1.0000        55

    accuracy                         1.0000       115
   macro avg     1.0000    1.0000    1.0000       115
weighted avg     1.0000    1.0000    1.0000       115

ROC AUC: 1.0


Celda 6 — Guardar modelo y reporte

In [7]:
os.makedirs(BASE_DIR, exist_ok=True)

joblib.dump({
    "model": rf,
    "feature_cols": feature_cols
}, OUT_MODEL)

with open(OUT_REPORT, "w", encoding="utf-8") as f:
    f.write("MODELO COMPARATIVO - RANDOM FOREST (BINARIO OK vs NO_OK)\n\n")
    f.write("CONFIG:\n")
    f.write("- n_estimators=600\n- class_weight=balanced\n- min_samples_leaf=2\n\n")

    f.write("FEATURES:\n")
    f.write(", ".join(feature_cols) + "\n\n")

    f.write("=== VALIDACIÓN ===\n")
    f.write(rep_val + "\n")
    f.write(f"ROC AUC: {auc_val}\n\n")

    f.write("=== TEST ===\n")
    f.write(rep_test + "\n")
    f.write(f"ROC AUC: {auc_test}\n\n")

    f.write("=== VALIDACIÓN CRUZADA 5-FOLD ===\n")
    f.write(f"Accuracy mean: {np.mean(cv_scores['test_accuracy'])}\n")
    f.write(f"Accuracy std : {np.std(cv_scores['test_accuracy'])}\n")
    f.write(f"F1 mean      : {np.mean(cv_scores['test_f1'])}\n")
    f.write(f"F1 std       : {np.std(cv_scores['test_f1'])}\n")
    f.write(f"Recall mean  : {np.mean(cv_scores['test_recall'])}\n")
    f.write(f"Recall std   : {np.std(cv_scores['test_recall'])}\n")
    f.write(f"ROC AUC mean : {np.mean(cv_scores['test_roc_auc'])}\n")
    f.write(f"ROC AUC std  : {np.std(cv_scores['test_roc_auc'])}\n\n")

print("Modelo guardado en:", OUT_MODEL)
print("Reporte guardado en:", OUT_REPORT)

Modelo guardado en: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 2\RANDOMFOREST_model.joblib
Reporte guardado en: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 2\RANDOMFOREST_rf_report.txt


Celda 7 — Importancias de features (para comparar con coeficientes de LogReg)

In [8]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances

imp_path = os.path.join(BASE_DIR, "rf_feature_importances.csv")
importances.to_csv(imp_path, header=["importance"])
print("Importancias guardadas en:", imp_path)

Importancias guardadas en: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 2\rf_feature_importances.csv


Celda 8 — Comparación rápida contra LogReg (manual)

(Esto solo imprime un resumen para que tú pegues tus métricas de LogReg y compares.)

In [9]:
print("=== RESUMEN RF (TEST) ===")
print("ROC AUC (TEST):", auc_test)
print("\nREPORTE (TEST):\n", rep_test)

print("\nSugerencia: comparar con LogReg usando mismas métricas:")
print("- Accuracy")
print("- Recall NO_OK (clase 1)")
print("- F1 NO_OK (clase 1)")
print("- ROC AUC")

=== RESUMEN RF (TEST) ===
ROC AUC (TEST): 1.0

REPORTE (TEST):
               precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        60
           1     1.0000    1.0000    1.0000        55

    accuracy                         1.0000       115
   macro avg     1.0000    1.0000    1.0000       115
weighted avg     1.0000    1.0000    1.0000       115


Sugerencia: comparar con LogReg usando mismas métricas:
- Accuracy
- Recall NO_OK (clase 1)
- F1 NO_OK (clase 1)
- ROC AUC
